In [0]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if project_root not in sys.path:
    sys.path.append(project_root)

from pyspark.sql.functions import lit, col, when, timestamp_diff
from modules.transformation.geography import haversine_distance
from modules.utils.date import get_months_start_n_months_ago

In [0]:
# Enrich the bike trip records by joining cleansed records and stations lookup
# 3. Load the latest cleansed bike trips records, and stations lookup table for geographic enrichment
# 5. Join trip records with station data on both start and end stations
# 6. Enrich trips with station names, coordinates, distance calculations, and trip duration
# 7. Append data to silver layer enriched table

target_month = int(dbutils.widgets.get("months_ago"))
target_month_start = get_months_start_n_months_ago(months_ago=target_month)

In [0]:
df_trips = spark.read.table("bikes.02_silver.trips_cleansed").filter(f"started_at >= '{target_month_start}'")
df_stations = spark.read.table("bikes.02_silver.stations_cleansed")

In [0]:
df_join_start = df_trips.join(
    df_stations,
    ((df_trips.start_station_id == df_stations.short_name) | 
    ((df_trips.start_lat == df_stations.latitude) & (df_trips.start_lng == df_stations.longitude))) &
    (df_stations.valid_to.isNull()),
    "left"
).select(
    df_trips.city,
    df_trips.ride_id,
    df_trips.rideable_type,
    df_trips.started_at,
    df_trips.ended_at,
    df_stations.name.alias("start_station_name"),
    df_trips.end_station_id,
    df_stations.latitude.alias("start_latitude"),
    df_stations.longitude.alias("start_longitude"),
    df_trips.end_lat,
    df_trips.end_lng,
    df_trips.member_casual,
    df_trips.processed_timestamp
)

In [0]:
df_join_end = df_join_start.join(
    df_stations,
    ((df_join_start.end_station_id == df_stations.short_name) |
    ((df_join_start.end_lat == df_stations.latitude) & (df_join_start.end_lng == df_stations.longitude))) &
    (df_stations.valid_to.isNull()),
    "left"
).select(
    df_join_start.city,
    df_join_start.ride_id,
    df_join_start.rideable_type,
    df_join_start.started_at,
    df_join_start.ended_at,
    timestamp_diff(lit("MINUTE"),df_join_start.started_at,df_join_start.ended_at).alias("trip_duration_mins"),
    df_join_start.start_station_name,
    df_stations.name.alias("end_station_name"),
    df_join_start.start_latitude,
    df_join_start.start_longitude,
    df_stations.latitude.alias("end_latitude"),
    df_stations.longitude.alias("end_longitude"),
    haversine_distance(df_join_start.start_latitude,df_join_start.start_longitude,df_stations.latitude,df_stations.longitude).alias("stations_distance"),
    df_trips.member_casual,
    df_trips.processed_timestamp
)

In [0]:
df_join_end.filter(df_join_end.stations_distance > 0).write.mode("append").saveAsTable("bikes.02_silver.trips_enriched")